In [6]:
import pandas as pd
import os

for dirname, _, filenames in os.walk('/kaggle/input/datasets/zusmani/palestine-body-count'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

df = pd.read_csv('/kaggle/input/datasets/zusmani/palestine-body-count/Palestine Body Count.csv') 
print(df.head())
print(df.info())

/kaggle/input/datasets/zusmani/palestine-body-count/Palestine Body Count.csv
     Year      Month Palestinians Injuries Israelis Injuries  \
0  2000.0   DECEMBER                   781               NaN   
1  2000.0   NOVEMBER                  3838               NaN   
2  2000.0    OCTOBER                  5984               NaN   
3  2000.0  SEPTEMBER                   NaN               NaN   
4  2001.0   DECEMBER                   304               NaN   

  Palestinians Killed Israelis Killed  
0                  51               8  
1                 112              22  
2                 104              10  
3                  16               1  
4                  67              36  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 251 entries, 0 to 250
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Year                   249 non-null    float64
 1   Month                  249 non-nu

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
import pickle

df = pd.read_csv('/kaggle/input/datasets/zusmani/palestine-body-count/Palestine Body Count.csv')

cols_to_clean = ['Palestinians Injuries', 'Israeli Injuries', 'Palestinians Killed', 'Israeli Killed']

for col in cols_to_clean:
    if col in df.columns:
        
        df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', ''), errors='coerce')

df = df.dropna(subset=['Year', 'Palestinians Killed'])

month_mapping = {'JANUARY':1, 'FEBRUARY':2, 'MARCH':3, 'APRIL':4, 'MAY':5, 'JUNE':6, 
                 'JULY':7, 'AUGUST':8, 'SEPTEMBER':9, 'OCTOBER':10, 'NOVEMBER':11, 'DECEMBER':12}
df['Month_Num'] = df['Month'].str.upper().map(month_mapping)
df = df.dropna(subset=['Month_Num'])

X = df[['Year', 'Month_Num']]
y = df['Palestinians Killed']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree Regressor": DecisionTreeRegressor(random_state=42),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting Regressor": GradientBoostingRegressor(random_state=42),
    "XGBoost Regressor": XGBRegressor(random_state=42)
}

model_performances = {}

print("--- Training & Evaluating Regression Models ---")
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    r2 = r2_score(y_test, y_pred)
    model_performances[name] = r2
    print(f"-> {name} R2 Score: {r2:.4f}")

best_model_name = max(model_performances, key=model_performances.get)
print(f"\n Best Performing Model: {best_model_name} with R2 Score: {model_performances[best_model_name]:.4f}\n")

best_model = models[best_model_name]
best_model.fit(X, y) 

with open('best_palestine_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

print(" Model successfully trained and saved as 'best_palestine_model.pkl'!")

--- Training & Evaluating Regression Models ---
-> Linear Regression R2 Score: 0.1172
-> Decision Tree Regressor R2 Score: 0.0453
-> Random Forest Regressor R2 Score: 0.3518
-> Gradient Boosting Regressor R2 Score: -0.1620
-> XGBoost Regressor R2 Score: 0.2818

🏆 Best Performing Model: Random Forest Regressor with R2 Score: 0.3518

 Model successfully trained and saved as 'best_palestine_model.pkl'!


In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import Ridge
import pickle

df = pd.read_csv('/kaggle/input/datasets/zusmani/palestine-body-count/Palestine Body Count.csv')
df.columns = df.columns.str.strip()

cols_to_clean = ['Palestinians Injuries', 'Israeli Injuries', 'Palestinians Killed', 'Israeli Killed']
for col in cols_to_clean:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', ''), errors='coerce').fillna(0)

month_mapping = {'JANUARY':1, 'FEBRUARY':2, 'MARCH':3, 'APRIL':4, 'MAY':5, 'JUNE':6, 
                 'JULY':7, 'AUGUST':8, 'SEPTEMBER':9, 'OCTOBER':10, 'NOVEMBER':11, 'DECEMBER':12}
df['Month_Num'] = df['Month'].str.strip().str.upper().map(month_mapping)

df = df.sort_values(by=['Year', 'Month_Num']).reset_index(drop=True)

df['Rolling_Killed_3M'] = df['Palestinians Killed'].rolling(window=3, min_periods=1).mean().fillna(0)
df['Rolling_Injured_3M'] = df['Palestinians Injuries'].rolling(window=3, min_periods=1).mean().fillna(0)
df['Lag_1_Killed'] = df['Palestinians Killed'].shift(1).fillna(0)
df['Lag_2_Killed'] = df['Palestinians Killed'].shift(2).fillna(0)

X = df[['Year', 'Month_Num', 'Rolling_Killed_3M', 'Rolling_Injured_3M', 'Lag_1_Killed', 'Lag_2_Killed']]
y = df['Palestinians Killed']

X = X.fillna(0)
y = y.fillna(0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False)


models = {
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=150, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "XGBoost Regressor": XGBRegressor(n_estimators=100, learning_rate=0.05, random_state=42)
}

best_score = -999
best_model_name = ""
best_model = None

print("--- Training Models without NaNs ---")
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    r2 = r2_score(y_test, y_pred)
    print(f"-> {name} R2 Score: {r2:.4f}")
    
    if r2 > best_score:
        best_score = r2
        best_model_name = name
        best_model = model

print(f"\n Best Model: {best_model_name} with R2 Score: {best_score:.4f}")


best_model.fit(X, y)
with open('best_palestine_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

print(" Model successfully trained and saved as 'best_palestine_model.pkl'!")

--- Training Models without NaNs ---
-> Ridge Regression R2 Score: 0.9969
-> Random Forest Regressor R2 Score: 0.1352
-> Gradient Boosting R2 Score: 0.1495
-> XGBoost Regressor R2 Score: 0.1619

🏆 Best Model: Ridge Regression with R2 Score: 0.9969
✅ Model successfully trained and saved as 'best_palestine_model.pkl'!


In [15]:
import pickle
import numpy as np
import pandas as pd

with open('best_palestine_model.pkl', 'rb') as f:
    model, imputer = pickle.load(f)

print(" Model & Imputer successfully loaded!")

sample_input = np.array([[2026, 10, 65.0, 40.0, 70.0, 85.0]])

sample_input_imputed = imputer.transform(sample_input)
prediction = model.predict(sample_input_imputed)

print(f" Predicted Casualties (Palestinians Killed): {int(prediction[0])}")

✅ Model & Imputer successfully loaded!
🎯 Predicted Casualties (Palestinians Killed): 39


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(


In [17]:
from IPython.display import FileLink

FileLink('best_palestine_model.pkl')

/kaggle/working/best_palestine_model.pkl